# STEP Files

Clear the OCP CAD Viewer first, then refresh and open the canonical `type2_scene.step` artifact in VS Code.


In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

import build123d as bd
from ocp_vscode import Camera, show, show_clear


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    repo_root = Path(root_text).resolve()
    pyproject_path = repo_root / "pyproject.toml"
    if not pyproject_path.is_file():
        raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
    return repo_root


REPO_ROOT = require_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from entry.refresh_type2_step_viewer_artifacts import refresh_type2_step_viewer_artifacts

TYPE2_SOURCE_TOML_PATH = REPO_ROOT / "examples" / "type2_fixed.toml"
TYPE2_STEP_OUTPUT_DIR = REPO_ROOT / "run" / "step" / "type2"
TYPE2_STEP_LEDGER_PATH = TYPE2_STEP_OUTPUT_DIR / "type2_step_ledger.json"
_COPPER_COLOR = (184, 115, 51)
_PCB_COLOR = (0, 128, 0)
_NON_MODEL_COLOR = (128, 128, 128)
_COPPER_ALPHA = 1.0
_PCB_ALPHA = 0.4
_NON_MODEL_ALPHA = 0.12


def viewer_style_from_label(label: str) -> tuple[tuple[int, int, int], float]:
    if label.startswith(("tx_copper_l", "rx_copper_l")):
        return (_COPPER_COLOR, _COPPER_ALPHA)
    if label.startswith(("tx_pcb_l", "rx_pcb_l")):
        return (_PCB_COLOR, _PCB_ALPHA)
    return (_NON_MODEL_COLOR, _NON_MODEL_ALPHA)


def child_shapes(shape: bd.Shape) -> list[bd.Shape]:
    children = tuple(shape.children)
    if children:
        return list(children)
    return [shape]


def viewer_payload_for_shape(shape: bd.Shape, *, fallback_name: str) -> tuple[list[bd.Shape], list[str], list[tuple[int, int, int]], list[float]]:
    entries = child_shapes(shape)
    cad_objs: list[bd.Shape] = []
    names: list[str] = []
    colors: list[tuple[int, int, int]] = []
    alphas: list[float] = []
    for index, entry in enumerate(entries):
        entry_label = entry.label if isinstance(entry.label, str) and entry.label != "" else f"{fallback_name}_{index}"
        color, alpha = viewer_style_from_label(entry_label)
        cad_objs.append(entry)
        names.append(entry_label)
        colors.append(color)
        alphas.append(alpha)
    return (cad_objs, names, colors, alphas)


print(f"repo root: {REPO_ROOT}")
print(f"type2 TOML: {TYPE2_SOURCE_TOML_PATH}")
print(f"STEP output dir: {TYPE2_STEP_OUTPUT_DIR}")
print(f"STEP ledger: {TYPE2_STEP_LEDGER_PATH}")
show_clear()
print("viewer cleared")


repo root: /home/harry/Projects/PythonProjects/peetsfea-main
type2 TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2_fixed.toml
STEP output dir: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2
STEP ledger: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
Using port 3939
viewer cleared


## Refresh and show scene STEP

This notebook clears the viewer first, refreshes the fixed `type2_scene.step` artifact, and opens that single scene file with a reset camera.


In [2]:
refresh_result = refresh_type2_step_viewer_artifacts(
    toml_path=TYPE2_SOURCE_TOML_PATH,
    output_dir=TYPE2_STEP_OUTPUT_DIR,
    ledger_path=TYPE2_STEP_LEDGER_PATH,
    seed=0,
)
scene_step_path = Path(refresh_result["scene_step_path"])
shown_step = bd.import_step(scene_step_path)
cad_objs, names, colors, alphas = viewer_payload_for_shape(shown_step, fallback_name="type2_scene")
show(
    *cad_objs,
    names=names,
    colors=colors,
    alphas=alphas,
    transparent=True,
    reset_camera=Camera.RESET,
)
print(f"scene STEP: {scene_step_path}")
print(f"ledger JSON: {refresh_result['ledger_path']}")


+++++++++
scene STEP: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_scene.step
ledger JSON: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
